# Conditional GAN (cGAN)

This notebook implements **Conditional GAN** (Mirza & Osindero, 2014), following Module 13.

Until now the Generator has been:

$$G(z) \rightarrow x$$

We can generate an image, but we cannot tell the Generator *what* to draw.

cGAN adds a condition $y$ — class label, attribute, text, even another image:

$$G(z, y) \rightarrow x$$

The central idea is that the Generator models a **conditional distribution**:

$$p_g(x \mid y)$$

rather than just $p_g(x)$. Given a class label $y$, the Generator produces a sample from that class.

Critically, the Discriminator is also conditioned:

$$D(x, y) \rightarrow \text{real/fake}$$

so it can learn *is this image realistic AND compatible with the requested condition?* — not just *is it realistic?*

We will keep the DCGAN backbone and add conditioning on MNIST's 10 digit classes.

## Implementation Plan

We use a simple, well-established cGAN recipe:

- **Class conditioning via learned `nn.Embedding`**
- **Generator**: concatenate $z$ and the label embedding $e_y$, then project to a 4×4 feature map and run the usual DCGAN `ConvTranspose2d` progression to 64×64.
- **Discriminator**: broadcast the label embedding to a 64×64 spatial map, treat it as an *extra input channel* alongside the image, and run the standard DCGAN `Conv2d` stack.
- **Loss**: standard `BCEWithLogitsLoss`, just like DCGAN. The condition enters both networks, not the loss.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

torch.manual_seed(42)
np.random.seed(42)

## 1. Setup and Hyperparameters

DCGAN defaults carry over: Adam, `lr=2e-4`, `betas=(0.5, 0.999)`, 25 epochs.

Two new pieces:

- `n_classes = 10` (MNIST digits)
- `embed_dim = 10` (size of each class's learned embedding)

So the Generator input is a concatenation of $z \in \mathbb{R}^{100}$ and $e_y \in \mathbb{R}^{10}$ — a 110-dim vector.

In [ ]:
latent_dim    = 100
n_classes     = 10
embed_dim     = 10
img_channels  = 1
img_size      = 64
features_g    = 64
features_d    = 64
batch_size    = 64
lr            = 2e-4
betas         = (0.5, 0.999)
epochs        = 25

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('samples_cGAN', exist_ok=True)

## 2. Data — MNIST with Labels

MNIST's `DataLoader` already returns `(image, label)` tuples — we just need to actually *use* the labels now. In DCGAN we ignored them with `for real_imgs, _ in dataloader`. Here we read them: `for real_imgs, real_labels in dataloader`.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

## 3. Weight Initialization

Same DCGAN init as before.

In [ ]:
def weights_init(m):
    """DCGAN-style weight initialization."""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 4. The Generator — `G(z, y)`

Three conceptual steps:

1. **Embed the label.** `nn.Embedding(n_classes, embed_dim)` maps each integer class to a learned 10-D vector $e_y$.
2. **Concatenate** $z$ and $e_y$: $\text{input} = [z; e_y] \in \mathbb{R}^{110}$.
3. **Project** the 110-D vector to a `4×4` feature map, then run the standard DCGAN upsampling stack.

Spatial flow (same as DCGAN — only the **input** changed):

```
110   ->   4 × 4   ->   8 × 8   ->   16 × 16   ->   32 × 32   ->   64 × 64
          features_g*8     256          128              64              1
```

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, n_classes=10, embed_dim=10,
                 img_channels=1, features_g=64):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, embed_dim)

        input_dim = latent_dim + embed_dim  # 110

        # Project (z, e_y) -> 4x4 feature map
        self.init = nn.Sequential(
            nn.Linear(input_dim, features_g * 8 * 4 * 4),
            nn.BatchNorm1d(features_g * 8 * 4 * 4),
            nn.ReLU(True),
        )

        # Standard DCGAN upsampling progression
        self.main = nn.Sequential(
            # 4 -> 8
            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),

            # 8 -> 16
            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),

            # 16 -> 32
            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),

            # 32 -> 64, project to img_channels, Tanh
            nn.ConvTranspose2d(features_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        l = self.label_emb(labels)            # (B, embed_dim)
        x = torch.cat([z, l], dim=1)          # (B, 110)
        x = self.init(x)                       # (B, features_g*8*4*4)
        x = x.view(x.size(0), -1, 4, 4)        # (B, features_g*8, 4, 4)
        return self.main(x)

## 5. The Discriminator — `D(x, y)`

Three steps again:

1. **Embed the label.** Same `nn.Embedding` — though here it's a *separate* embedding with its own parameters, not shared with $G$.
2. **Broadcast to a spatial map.** A linear layer maps $e_y \in \mathbb{R}^{10}$ to a 64×64 feature map. We treat that map as an *extra channel* alongside the image.
3. **Run the standard DCGAN stack** on the 2-channel input.

Why broadcast to a full-resolution map and not just concatenate at the bottleneck? Both work. Using a full-resolution spatial map lets every convolutional filter see the label, which means the Discriminator can detect condition–content mismatches at any spatial scale. It costs one extra channel of compute.

Spatial flow:

```
64 × 64  ->  32 × 32  ->  16 × 16  ->   8 × 8   ->   4 × 4   ->  1
    2         64         128          256           512        1
```

Note the **first conv layer has 2 input channels** (image + label map) instead of 1. We still keep no BatchNorm in the first layer — the DCGAN rule carries over.

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=1, n_classes=10, embed_dim=10,
                 features_d=64, img_size=64):
        super().__init__()
        self.img_size = img_size
        self.label_emb = nn.Embedding(n_classes, embed_dim)

        # Project label embedding to a (1, img_size, img_size) spatial map.
        # We then concatenate this with the image as an extra channel.
        self.label_proj = nn.Linear(embed_dim, img_size * img_size)

        # 2-channel input: 1 image channel + 1 label-channel
        self.net = nn.Sequential(
            # 64 -> 32, NO BatchNorm in the first block
            nn.Conv2d(img_channels + 1, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 32 -> 16
            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 16 -> 8
            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 8 -> 4
            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 4 -> 1 (logit) — no Sigmoid, BCEWithLogitsLoss
            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x, labels):
        l = self.label_emb(labels)                              # (B, embed_dim)
        l = self.label_proj(l)                                  # (B, img_size*img_size)
        l = l.view(l.size(0), 1, self.img_size, self.img_size)  # (B, 1, H, W)
        x = torch.cat([x, l], dim=1)                            # (B, 2, H, W)
        return self.net(x).view(x.size(0), -1)

## 6. Models, Optimizers, and Loss

Loss is identical to DCGAN: `BCEWithLogitsLoss`. cGAN does not introduce a new loss — the conditioning is fully absorbed into the networks.

In [ ]:
G = Generator(
    latent_dim=latent_dim,
    n_classes=n_classes,
    embed_dim=embed_dim,
    img_channels=img_channels,
    features_g=features_g,
).to(device)
D = Discriminator(
    img_channels=img_channels,
    n_classes=n_classes,
    embed_dim=embed_dim,
    features_d=features_d,
    img_size=img_size,
).to(device)

G.apply(weights_init)
D.apply(weights_init)

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)

criterion = nn.BCEWithLogitsLoss()

print(G)
print()
print(D)

## 7. Fixed Noise for Visualization

For DCGAN we made a `fixed_noise` of 64 random vectors. Here we want to *see* the conditioning, so we make a **fixed (noise, label) grid**:

- 10 rows (one per digit class)
- 8 columns (8 different noise vectors per class)
- The first 8 vectors produce 8 versions of "0", the next 8 produce 8 versions of "1", and so on.

This grid is the **direct visual check that conditioning works**: each row should look like the corresponding digit.

In [ ]:
n_per_class = 8
fixed_noise  = torch.randn(n_classes * n_per_class, latent_dim, device=device)
fixed_labels = torch.tensor(
    [c for c in range(n_classes) for _ in range(n_per_class)],
    device=device,
    dtype=torch.long,
)

print('fixed_noise :', tuple(fixed_noise.shape))
print('fixed_labels:', tuple(fixed_labels.shape))

## 8. The Training Loop

Same alternating structure as DCGAN, with one change: **labels travel with both real images and generated images**.

**Phase A — Train D**

1. Real images and their labels $\rightarrow D \rightarrow$ logit, target $1$
2. Fake images and the *same* labels $\rightarrow D \rightarrow$ logit, target $0$ (detached)
3. BCE loss, backprop into D only

**Phase B — Train G**

1. Generate fakes from $z$ and labels (no detach)
2. $D(\text{fakes}, y) \rightarrow$ logit, target $1$
3. BCE loss, backprop through D into G

The Discriminator's gradient now carries information about whether the *content* matches the *label*. That is the entire mechanism by which $G$ learns to obey $y$.

In [ ]:
losses_g = []
losses_d = []

G.train()
D.train()

for epoch in range(epochs):
    d_running_loss = 0.0
    g_running_loss = 0.0
    n_batches = 0

    for real_imgs, real_labels in dataloader:
        real_imgs = real_imgs.to(device)
        real_labels = real_labels.to(device)
        b = real_imgs.size(0)

        valid = torch.ones(b, 1, device=device)
        fake   = torch.zeros(b, 1, device=device)

        # -----------------
        # Phase A: Train D
        # -----------------
        optimizer_D.zero_grad()

        # Real images + their labels -> label 1
        real_logits = D(real_imgs, real_labels)
        d_real_loss = criterion(real_logits, valid)

        # Fake images + same labels -> label 0, detached
        z = torch.randn(b, latent_dim, device=device)
        with torch.no_grad():
            gen_imgs = G(z, real_labels)
        fake_logits = D(gen_imgs, real_labels)
        d_fake_loss = criterion(fake_logits, fake)

        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # -----------------
        # Phase B: Train G
        # -----------------
        optimizer_G.zero_grad()

        gen_imgs = G(z, real_labels)              # NO detach
        validity_logits = D(gen_imgs, real_labels)
        g_loss = criterion(validity_logits, valid)

        g_loss.backward()
        optimizer_G.step()

        d_running_loss += d_loss.item()
        g_running_loss += g_loss.item()
        n_batches += 1

    avg_d = d_running_loss / n_batches
    avg_g = g_running_loss / n_batches
    losses_d.append(avg_d)
    losses_g.append(avg_g)

    print(f"Epoch [{epoch+1}/{epochs}]  D: {avg_d:.4f}  G: {avg_g:.4f}")

    # Save a per-class sample grid every epoch
    G.eval()
    with torch.no_grad():
        sample_imgs = G(fixed_noise, fixed_labels).detach().cpu()
    save_image(
        sample_imgs,
        f"samples_cGAN/epoch_{epoch+1:02d}.png",
        nrow=n_per_class,
        normalize=True,
    )
    G.train()

print('Training done.')

## 9. Class-Conditional Generation

This is the payoff: pick a class, generate as many variations of it as you want by varying only $z$. We make a 10×8 grid where row $i$ is 8 samples of digit $i$.

In [ ]:
G.eval()
with torch.no_grad():
    samples = G(fixed_noise, fixed_labels).cpu()

grid = make_grid(samples, nrow=n_per_class, normalize=True)
plt.figure(figsize=(10, 12))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('cGAN samples — rows are classes 0–9, columns are different z\'s')
plt.show()

## 10. Training Progression

Each saved grid is a full 10×8 class grid from a given epoch. As training proceeds, each row should consolidate into one digit class.

In [ ]:
from PIL import Image
import glob

paths = sorted(glob.glob('samples_cGAN/epoch_*.png'))
if paths:
    fig, axes = plt.subplots(1, len(paths), figsize=(2.2 * len(paths), 2.2))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p)).squeeze(), cmap='gray')
        ax.set_title(p.split('_')[-1].split('.')[0])
        ax.axis('off')
    plt.suptitle('Generator progression across epochs — rows are classes 0–9')
    plt.show()
else:
    print('No sample grids found. Run the training cell first.')

## 11. Loss Curves

Same BCE dynamics as DCGAN. The conditioning did not change the loss surfaces meaningfully — only the data each network sees.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses_d, label='Discriminator')
plt.plot(losses_g, label='Generator')
plt.xlabel('Epoch')
plt.ylabel('BCELoss')
plt.title('cGAN losses')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Recap — What cGAN Adds on Top of DCGAN

| Concern | DCGAN | cGAN |
|---|---|---|
| Generator input | $z$ | $[z; e_y]$ |
| Discriminator input | $x$ | $x$ + label as extra channel |
| Class info in loss? | n/a | **No** — only the networks see it |
| Distribution modeled | $p_g(x)$ | $p_g(x \mid y)$ |
| Inference control | none — sample is whatever G draws | **explicit** — pick $y$, vary $z$ for diversity |
| Loss | `BCEWithLogitsLoss` | same |
| Optimizer / LR | Adam, 2e-4 | same |
| Architecture | DCGAN | same DCGAN |
| Mode collapse | can still occur | can still occur **within** a class |

The architecture barely changed. The Generator gets a 110-D input instead of 100-D. The Discriminator's first conv sees 2 channels instead of 1. That's it.

But the **capability** is fundamentally different: cGAN models $p(x \mid y)$, not $p(x)$. We can now ask "draw me a 7" and get a 7 back, with $z$ controlling which 7.

**Common implementation pitfalls** — quick reference:

- **Labels must travel everywhere.** Every call to $G$ and $D$ needs a label tensor of the right shape `(B,)`.
- **Don't accidentally condition $D$ on the wrong label.** If you give the Discriminator real images with their real labels but fake images with random labels, you're teaching $D$ that *any* mismatch is fake. That is too strong and stalls training. Use the **same** label for the real and the fake branch of each batch.
- **Don't forget to use labels in the training loop.** The single most common bug is leaving `for real_imgs, _ in dataloader` from DCGAN — the underscore silently throws the labels away.
- **`fixed_labels` ordering matters** if you want a per-class grid. Tile carefully: `[c] * n_per_class` for each $c$.
- **cGAN does not eliminate mode collapse.** It shifts the problem: now $G$ can collapse *within* a class — every $z$ in the "cat" condition gives the same cat. Diversity and conditional control are separate concerns.

**Next in the series** (Module 14): ACGAN, where the Discriminator grows a *second head* that explicitly predicts the class. Instead of merely asking "is this image realistic given $y$?", ACGAN's Discriminator also asks "what class is this?" — and the Generator is rewarded twice for getting it right.